# 4. NinjaTrader 8 Implementation - ES 15min Mean Reversion Strategy
## Production-Ready Script for Prop Firm / Funded Account

This notebook contains the final NinjaTrader 8 implementation of the ES 15min Mean Reversion strategy.

**Strategy Parameters (Optimized & Validated):**
- Entry Threshold: 1.0% (price move)
- RSI(2) Threshold: 16 (oversold) / 84 (overbought)
- ATR Stop Multiplier: 1.4x
- ATR Target Multiplier: 3.0x
- Time Stop: 3:50 PM NY
- Volume Ratio: > 1.2x
- Day Filter: Tuesday-Friday
- Time Window: 9:30 AM - 3:30 PM NY

**Inheritance:** The strategy inherits from `HealthMonitor` (which inherits from `PerformanceReporter`), providing:
- Health monitoring (slippage, order rejects, frozen orders, partial fills, etc.)
- Performance reporting to backend
- Contract rollover alerts
- Inactivity alerts
- Connection status monitoring

**Key Features:**
- Dynamic ATR-based stop-loss and take-profit
- Time stop at 3:50 PM NY (no open positions after 4 PM)
- Volume confirmation
- RSI(2) oversold/overbought conditions
- Day and time filters
- Position sizing via UI parameter (Quantity)
- Real-time trade reporting to backend

In [ ]:
// MR_15min_MeanReversion.cs
using System;
using System.ComponentModel.DataAnnotations;
using System.Windows.Media;
using NinjaTrader.NinjaScript.Indicators;
using NinjaTrader.NinjaScript.Strategies;

namespace NinjaTrader.NinjaScript.Strategies
{
    public class MR_15min_MeanReversion : HealthMonitor
    {
        // ================================ INDICATORS ================================
        private RSI rsi2;
        private ATR atr14;
        private SMA volumeMA20;
        private SMA priceMA200; // Optional trend filter (commented out)

        // ================================ STATE TRACKING ================================
        private bool hasPosition;
        private DateTime entryTime;
        private double entryPrice;
        private double atrValue;

        // ================================ INITIALIZATION ================================
        protected override void OnStateChange()
        {
            if (State == State.SetDefaults)
            {
                // --- Basic strategy information ---
                Name = "MR_15min_MeanReversion";
                Description = "ES 15min Mean Reversion with full health monitoring.";

                // --- Strategy parameters ---
                Quantity = 1;
                EntryThresholdPercent = 1.0;
                RsiThreshold = 16;
                AtrStopMultiplier = 1.4;
                AtrTargetMultiplier = 3.0;
                VolumeRatioThreshold = 1.2;
                TimeStopHour = 15;
                TimeStopMinute = 50;

                // --- Health monitoring (inherited) ---
                MaxBarsWithoutTrade = 50;
                EnableInactivityAlert = true;
                RolloverAlertDays = 5;

                // --- Technical settings ---
                Calculate = Calculate.OnBarClose;
                IsInstantiatedOnEachOptimizationIteration = false;
            }
            else if (State == State.DataLoaded)
            {
                // --- Initialize indicators ---
                // RSI(2) - extreme short-term overbought/oversold
                rsi2 = RSI(2, 0);
                rsi2.Plots[0].Brush = Brushes.Magenta;
                AddChartIndicator(rsi2);

                // ATR(14) - for dynamic stops
                atr14 = ATR(14);
                atr14.Plots[0].Brush = Brushes.Orange;
                AddChartIndicator(atr14);

                // Volume MA(20) - for volume confirmation
                volumeMA20 = SMA(Volume, 20);
                volumeMA20.Plots[0].Brush = Brushes.Gray;
                volumeMA20.Plots[0].Width = 2;
                AddChartIndicator(volumeMA20);

                // Optional: 200-period MA for trend filter (commented out)
                // priceMA200 = SMA(Close, 200);
                // priceMA200.Plots[0].Brush = Brushes.DarkCyan;
                // AddChartIndicator(priceMA200);

                hasPosition = false;
                entryTime = DateTime.MinValue;
                entryPrice = 0;
                atrValue = 0;
            }

            // ✅ IMPORTANT: Always call base.OnStateChange() for health monitoring & rollover
            base.OnStateChange();
        }

        // ================================ MAIN TRADING LOGIC ================================
        protected override void OnBarUpdate()
        {
            // ✅ IMPORTANT: Call base.OnBarUpdate() for rollover detection and health monitoring
            base.OnBarUpdate();

            // --- Wait for enough bars ---
            if (CurrentBar < BarsRequiredToTrade) return;

            // --- Check if we have valid indicator values ---
            if (rsi2 == null || atr14 == null || volumeMA20 == null) return;
            if (rsi2[0] == 0 || double.IsNaN(rsi2[0])) return;
            if (atr14[0] == 0 || double.IsNaN(atr14[0])) return;
            if (volumeMA20[0] == 0 || double.IsNaN(volumeMA20[0])) return;

            // --- TIME FILTER: Only between 9:30 AM and 3:30 PM NY ---
            DateTime now = Time[0];
            int hour = now.Hour;
            int minute = now.Minute;

            // Only trade between 9:30 AM and 3:30 PM NY
            bool isWithinTimeWindow = (hour > 9 || (hour == 9 && minute >= 30)) && (hour < 15 || (hour == 15 && minute <= 30));

            // --- DAY FILTER: Only Tuesday-Friday ---
            bool isAllowedDay = (now.DayOfWeek != DayOfWeek.Monday && now.DayOfWeek != DayOfWeek.Saturday && now.DayOfWeek != DayOfWeek.Sunday);

            // --- TIME STOP: Close all positions by 3:50 PM NY ---
            bool isTimeStop = (hour > TimeStopHour || (hour == TimeStopHour && minute >= TimeStopMinute));

            // ============================================================
            // 1. EXIT CONDITIONS
            // ============================================================
            if (Position.MarketPosition != MarketPosition.Flat)
            {
                // --- 1a. TIME STOP ---
                if (isTimeStop)
                {
                    if (Position.MarketPosition == MarketPosition.Long)
                        ExitLong("TimeStop", "LongEntry");
                    else
                        ExitShort("TimeStop", "ShortEntry");
                    return;
                }

                // --- 1b. TRACK POSITION FOR DYNAMIC STOPS ---
                // If we already have a position, the SetStopLoss/SetProfitTarget handles exits.
                // ATR-based stops are set at entry time.
                return;
            }

            // ============================================================
            // 2. ENTRY CONDITIONS
            // ============================================================

            // --- Skip if outside time window or day filter ---
            if (!isWithinTimeWindow || !isAllowedDay) return;

            // --- Calculate required values ---
            double closePctChange = (Close[0] - Close[1]) / Close[1] * 100.0;
            double volumeRatio = Volume[0] / volumeMA20[0];
            double rsiValue = rsi2[0];
            atrValue = atr14[0];
            double entryThreshold = EntryThresholdPercent;

            // --- Optional trend filter (commented out) ---
            // bool isBearMarket = Close[0] < priceMA200[0];

            // --- LONG ENTRY: Drop >1.0%, RSI(2) < 16, Volume > 1.2x MA, within time/day ---
            if (closePctChange < -entryThreshold &&
                rsiValue < RsiThreshold &&
                volumeRatio > VolumeRatioThreshold)
            {
                // Enter long with ATR-based stop and target
                SetStopLoss(CalculationMode.Price, Close[0] - (atrValue * AtrStopMultiplier));
                SetProfitTarget(CalculationMode.Price, Close[0] + (atrValue * AtrTargetMultiplier));

                entryPrice = Close[0];
                entryTime = Time[0];
                hasPosition = true;
                EnterLong(Quantity, "LongEntry");
                Print($"📈 LONG Entry: Price={entryPrice:F2}, RSI={rsiValue:F2}, VolRatio={volumeRatio:F2}, Stop={Close[0] - (atrValue * AtrStopMultiplier):F2}, Target={Close[0] + (atrValue * AtrTargetMultiplier):F2}");
                return;
            }

            // --- SHORT ENTRY: Rise >1.0%, RSI(2) > 84, Volume > 1.2x MA, within time/day ---
            if (closePctChange > entryThreshold &&
                rsiValue > (100 - RsiThreshold) &&
                volumeRatio > VolumeRatioThreshold)
            {
                // Enter short with ATR-based stop and target
                SetStopLoss(CalculationMode.Price, Close[0] + (atrValue * AtrStopMultiplier));
                SetProfitTarget(CalculationMode.Price, Close[0] - (atrValue * AtrTargetMultiplier));

                entryPrice = Close[0];
                entryTime = Time[0];
                hasPosition = true;
                EnterShort(Quantity, "ShortEntry");
                Print($"📉 SHORT Entry: Price={entryPrice:F2}, RSI={rsiValue:F2}, VolRatio={volumeRatio:F2}, Stop={Close[0] + (atrValue * AtrStopMultiplier):F2}, Target={Close[0] - (atrValue * AtrTargetMultiplier):F2}");
                return;
            }
        }

        // ============================================================
        // 3. POSITION CLOSING - RESET STATE
        // ============================================================
        protected override void OnPositionUpdate(Position position, double averagePrice, int quantity, MarketPosition marketPosition)
        {
            // When position becomes flat, reset tracking
            if (quantity == 0 && hasPosition)
            {
                hasPosition = false;
                entryPrice = 0;
                entryTime = DateTime.MinValue;
                atrValue = 0;

                // Reset stop/target for next trade
                SetStopLoss(CalculationMode.Ticks, 0);
                SetProfitTarget(CalculationMode.Ticks, 0);
            }

            base.OnPositionUpdate(position, averagePrice, quantity, marketPosition);
        }

        // ============================================================
        // 4. UI-EDITABLE PARAMETERS
        // ============================================================

        // --- Order Management ---
        [Range(1, 100)]
        [NinjaScriptProperty]
        [Display(Name = "Quantity", Description = "Number of contracts per order", Order = 1, GroupName = "Order Management")]
        public int Quantity { get; set; }

        // --- Strategy Parameters ---
        [Range(0.5, 3.0)]
        [NinjaScriptProperty]
        [Display(Name = "Entry Threshold (%)", Description = "Minimum price move % to trigger entry", Order = 2, GroupName = "Strategy Parameters")]
        public double EntryThresholdPercent { get; set; }

        [Range(10, 25)]
        [NinjaScriptProperty]
        [Display(Name = "RSI(2) Threshold", Description = "RSI(2) oversold/overbought threshold (e.g., 16 = oversold, 84 = overbought)", Order = 3, GroupName = "Strategy Parameters")]
        public int RsiThreshold { get; set; }

        [Range(0.8, 2.0)]
        [NinjaScriptProperty]
        [Display(Name = "Volume Ratio Threshold", Description = "Minimum volume ratio vs 20-period average", Order = 4, GroupName = "Strategy Parameters")]
        public double VolumeRatioThreshold { get; set; }

        // --- ATR-based stops ---
        [Range(0.5, 3.0)]
        [NinjaScriptProperty]
        [Display(Name = "ATR Stop Multiplier", Description = "Stop loss = entry ± (ATR × multiplier)", Order = 5, GroupName = "Risk Management")]
        public double AtrStopMultiplier { get; set; }

        [Range(1.5, 5.0)]
        [NinjaScriptProperty]
        [Display(Name = "ATR Target Multiplier", Description = "Take profit = entry ± (ATR × multiplier)", Order = 6, GroupName = "Risk Management")]
        public double AtrTargetMultiplier { get; set; }

        // --- Time Stop ---
        [Range(0, 23)]
        [NinjaScriptProperty]
        [Display(Name = "Time Stop Hour", Description = "Hour (NY time) to force close all positions", Order = 7, GroupName = "Time Management")]
        public int TimeStopHour { get; set; }

        [Range(0, 59)]
        [NinjaScriptProperty]
        [Display(Name = "Time Stop Minute", Description = "Minute (NY time) to force close all positions", Order = 8, GroupName = "Time Management")]
        public int TimeStopMinute { get; set; }

        // ============================================================
        // 5. OPTIONAL: TREND FILTER (commented out)
        // ============================================================
        /*
        [NinjaScriptProperty]
        [Display(Name = "Enable Trend Filter", Description = "Only trade in bear market (below MA200)", Order = 9, GroupName = "Strategy Parameters")]
        public bool EnableTrendFilter { get; set; }
        */
    }
}

# 5. Deployment Instructions

## File Placement
1. Copy `MR_15min_MeanReversion.cs` to: Documents\NinjaTrader 8\bin\Custom\Strategies\
2. Compile in NinjaTrader:
- Go to **New → NinjaScript Editor**
- Right-click → **Compile**

## Strategy Setup
1. In NinjaTrader, open a chart with **ES 15-min** bars.
2. Right-click → **Strategies** → **Add Strategy**.
3. Select **MR_15min_MeanReversion**.
4. Configure parameters:
- **Quantity**: 1 (or adjust for your risk tolerance)
- **Entry Threshold (%)**: 1.0
- **RSI(2) Threshold**: 16
- **Volume Ratio Threshold**: 1.2
- **ATR Stop Multiplier**: 1.4
- **ATR Target Multiplier**: 3.0
- **Time Stop Hour**: 15
- **Time Stop Minute**: 50

5. **Backend URL**: Leave as configured (your monitoring backend).
6. **Auth Token**: Leave as configured.

## Health Monitoring (Inherited)
- **Max Bars Without Trade**: 50 (alerts if no trades)
- **Enable Inactivity Alert**: true
- **Rollover Alert (days)**: 5

## Risk Notes (for your $150k funded account)
- **1 contract** = ~$200 risk per trade (with 4-point stop)
- **Max loss streak of 7**: -$1,400 (26.7% of $5,250 drawdown limit)
- **Expected monthly return**: ~$400-$600 with 1 contract

## Next Steps
1. Test in **Simulation** mode for 3-6 months.
2. Monitor metrics via your backend dashboard.
3. After validation, enable in **Live** mode (start with 1 contract).
4. Add more strategies (trend following, etc.) to build a diversified portfolio.